In [1]:
!nvidia-smi
!python --version

Thu Apr 16 18:52:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip -q uninstall -y torch torchvision torchaudio vllm
!pip -q install -U uv
!uv pip install vllm --torch-backend=auto
!pip -q install transformers accelerate bitsandbytes fastapi uvicorn psutil GPUtil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 112.8 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 174 packages in 2.21s
Prepared 73 packages in 30.92s
Uninstalled 8 packages in 441ms
Installed 73 packages in 288ms
 + anthropic==0.96.0
 + apache-tvm-ffi==0.1.10
 + astor==0.8.1
 + blake3==1.0.8
 + cbor2==5.9.0
 + compressed-tensors==0.14.0.1
 - cuda-bindings==12.9.4
 + cuda-bindings==13.0.3
 - cuda-python==12.9.4
 + cuda-python==13.0.3
 + depyf==0.20.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.17.0
 + fastar==0.11.0
 + flashinfer-cubin==0.6.6
 + flashinfer-python==0.6.6
 + gguf==0.18.0
 - huggingface-hub==1.10.1
 + huggingface-hub==0.36.2
 + ijson==3.5.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.3.0
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.11.3
 + loguru==0.7.3
 + mistral-common==1.11.0
 + model-hosting-container-sta

In [3]:
import torch, vllm, transformers

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("vLLM:", vllm.__version__)
print("Transformers:", transformers.__version__)

PyTorch: 2.10.0+cu130
CUDA available: True
GPU count: 1
GPU: NVIDIA A100-SXM4-40GB
vLLM: 0.19.0
Transformers: 4.57.6


In [4]:
!nohup vllm serve Qwen/Qwen3-8B \
  --dtype bfloat16 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.9 \
  > vllm.log 2>&1 &

In [9]:
!tail -n 200 vllm.log

2026-04-16 18:53:43.262124: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-16 18:53:43.281534: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776365623.313742    1163 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776365623.322374    1163 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776365623.340342    1163 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [10]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen3-8B","object":"model","created":1776365814,"owned_by":"vllm","root":"Qwen/Qwen3-8B","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-b13bbb58f28d87c7","object":"model_permission","created":1776365814,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [11]:
import requests

url = "http://localhost:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen3-8B",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write 3 short bullet points about A100 GPUs."}
    ],
    "temperature": 0.7,
    "max_tokens": 128
}

try:
    resp = requests.post(url, json=payload, timeout=60)
    resp.raise_for_status()

    data = resp.json()
    print(resp.status_code)
    print(data)
    print(data["choices"][0]["message"]["content"])

except Exception as e:
    print("ERROR:", e)

200
{'id': 'chatcmpl-9f8697c6c1017b6f', 'object': 'chat.completion', 'created': 1776365827, 'model': 'Qwen/Qwen3-8B', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "<think>\nOkay, the user wants three short bullet points about A100 GPUs. Let me start by recalling what I know about the A100. It's part of NVIDIA's A-series, right? So first, I should mention its primary use cases. I remember that the A100 is used for AI training and high-performance computing. That's a key point.\n\nNext, the architecture. The A100 uses the Ampere architecture, which is important for its performance. Also, it has Tensor Cores, which are crucial for accelerating AI workloads. Maybe I should mention the 80GB", 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None}, 'logprobs': None, 'finish_reason': 'length', 'stop_reason': None, 'token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'prompt_tokens': 3

In [13]:
import time
import json
import statistics
import requests

BASE_URL = "http://localhost:8000/v1"
MODEL = "Qwen/Qwen3-8B"

In [14]:
def run_inference(prompt, max_tokens=128, temperature=0.7):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    start = time.time()
    resp = requests.post(f"{BASE_URL}/chat/completions", json=payload, timeout=120)
    resp.raise_for_status()
    latency = time.time() - start

    data = resp.json()
    usage = data.get("usage", {})
    content = data["choices"][0]["message"]["content"]

    completion_tokens = usage.get("completion_tokens", 0)
    prompt_tokens = usage.get("prompt_tokens", 0)
    total_tokens = usage.get("total_tokens", prompt_tokens + completion_tokens)

    metrics = {
        "latency_sec": round(latency, 3),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "tokens_per_sec": round(completion_tokens / latency, 2) if latency > 0 else 0.0,
    }

    return content, metrics, data

In [15]:
def benchmark_prompts(prompts, runs=3, max_tokens=128, temperature=0.7):
    results = []

    for run in range(runs):
        print(f"Run {run + 1}/{runs}")
        for i, prompt in enumerate(prompts, 1):
            content, metrics, _ = run_inference(
                prompt,
                max_tokens=max_tokens,
                temperature=temperature,
            )
            results.append({
                "run": run + 1,
                "prompt_index": i,
                "prompt": prompt,
                **metrics,
            })
            print(
                f"  Prompt {i}: "
                f"{metrics['latency_sec']:.3f}s, "
                f"{metrics['completion_tokens']} output tokens, "
                f"{metrics['tokens_per_sec']:.2f} tok/s"
            )

    summary = {
        "avg_latency_sec": round(statistics.mean(r["latency_sec"] for r in results), 3),
        "avg_tokens_per_sec": round(statistics.mean(r["tokens_per_sec"] for r in results), 2),
        "avg_completion_tokens": round(statistics.mean(r["completion_tokens"] for r in results), 2),
        "num_requests": len(results),
    }

    return results, summary


In [16]:
def show_one_test(prompt):
    content, metrics, data = run_inference(prompt)
    print("Reply:\n")
    print(content)
    print("\nMetrics:\n")
    print(json.dumps(metrics, indent=2))
    return data

In [17]:
show_one_test("Write 3 short bullet points about A100 GPUs.")

Reply:

<think>
Okay, the user wants three short bullet points about A100 GPUs. Let me start by recalling what I know about the A100. It's a high-end GPU from NVIDIA, part of the Ampere architecture. First, I should mention its key features. The A100 has a large number of CUDA cores, which is important for parallel processing. Then there's the memory capacity; I think it comes with 40GB or 80GB of HBM2 memory. That's a big deal for AI workloads. Also, the Tensor Core technology is crucial for accelerating AI and machine

Metrics:

{
  "latency_sec": 1.738,
  "prompt_tokens": 32,
  "completion_tokens": 128,
  "total_tokens": 160,
  "tokens_per_sec": 73.65
}


{'id': 'chatcmpl-8fdd1dd131bb37c2',
 'object': 'chat.completion',
 'created': 1776366187,
 'model': 'Qwen/Qwen3-8B',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "<think>\nOkay, the user wants three short bullet points about A100 GPUs. Let me start by recalling what I know about the A100. It's a high-end GPU from NVIDIA, part of the Ampere architecture. First, I should mention its key features. The A100 has a large number of CUDA cores, which is important for parallel processing. Then there's the memory capacity; I think it comes with 40GB or 80GB of HBM2 memory. That's a big deal for AI workloads. Also, the Tensor Core technology is crucial for accelerating AI and machine",
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': [],
    'reasoning': None},
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None,
   'token_ids': None}],
 'service_tier': None,
 'system_fingerprint': N

In [18]:
test_prompts = [
    "Write 3 short bullet points about A100 GPUs.",
    "Explain what tensor parallelism is in 3 sentences.",
    "Give a short summary of vLLM.",
]

results, summary = benchmark_prompts(test_prompts, runs=3, max_tokens=128)
print(json.dumps(summary, indent=2))

Run 1/3
  Prompt 1: 1.734s, 128 output tokens, 73.82 tok/s
  Prompt 2: 1.849s, 128 output tokens, 69.21 tok/s
  Prompt 3: 1.719s, 128 output tokens, 74.47 tok/s
Run 2/3
  Prompt 1: 1.719s, 128 output tokens, 74.48 tok/s
  Prompt 2: 1.718s, 128 output tokens, 74.49 tok/s
  Prompt 3: 1.719s, 128 output tokens, 74.48 tok/s
Run 3/3
  Prompt 1: 1.719s, 128 output tokens, 74.48 tok/s
  Prompt 2: 1.719s, 128 output tokens, 74.48 tok/s
  Prompt 3: 1.719s, 128 output tokens, 74.48 tok/s
{
  "avg_latency_sec": 1.735,
  "avg_tokens_per_sec": 73.82,
  "avg_completion_tokens": 128,
  "num_requests": 9
}


### AWQ 4-bit

In [26]:
!nohup vllm serve Qwen/Qwen3-8B-AWQ \
  --dtype auto \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.6 \
  > vllm.log 2>&1 &

In [29]:
!tail -n 200 vllm.log

# !sleep 10 && tail -n 100 vllm.log

2026-04-16 19:14:30.999444: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-16 19:14:31.018926: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776366871.041438    7344 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776366871.049067    7344 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776366871.067029    7344 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [30]:
BASE_URL = "http://localhost:8000/v1"
MODEL = "Qwen/Qwen3-8B-AWQ"

In [31]:
show_one_test("Write 3 short bullet points about A100 GPUs.")

Reply:

<think>
Okay, the user wants three short bullet points about A100 GPUs. Let me start by recalling what I know about A100. First, the A100 is part of NVIDIA's H100 series, right? Wait, no, actually, the A100 is part of the A1000 series. Wait, maybe I should check that. Wait, no, the A100 is part of the A1000 series, but I think the H100 is the newer one. Wait, maybe I'm mixing up the generations. Let me make

Metrics:

{
  "latency_sec": 0.959,
  "prompt_tokens": 32,
  "completion_tokens": 128,
  "total_tokens": 160,
  "tokens_per_sec": 133.43
}


{'id': 'chatcmpl-b2683e95ad13d277',
 'object': 'chat.completion',
 'created': 1776367110,
 'model': 'Qwen/Qwen3-8B-AWQ',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "<think>\nOkay, the user wants three short bullet points about A100 GPUs. Let me start by recalling what I know about A100. First, the A100 is part of NVIDIA's H100 series, right? Wait, no, actually, the A100 is part of the A1000 series. Wait, maybe I should check that. Wait, no, the A100 is part of the A1000 series, but I think the H100 is the newer one. Wait, maybe I'm mixing up the generations. Let me make",
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': [],
    'reasoning': None},
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None,
   'token_ids': None}],
 'service_tier': None,
 'system_fingerprint': None,
 'usage': {'prompt_tokens': 32,
  'total_tokens': 160,
  'completion_tokens': 128,
  'prompt_token

In [33]:
test_prompts = [
    "Write 3 short bullet points about A100 GPUs.",
    "Explain what tensor parallelism is in 3 sentences.",
    "Give a short summary of vLLM.",
]

results, summary = benchmark_prompts(test_prompts, runs=3, max_tokens=128)
print(json.dumps(summary, indent=2))

Run 1/3
  Prompt 1: 0.844s, 128 output tokens, 151.66 tok/s
  Prompt 2: 0.814s, 128 output tokens, 157.23 tok/s
  Prompt 3: 0.812s, 128 output tokens, 157.63 tok/s
Run 2/3
  Prompt 1: 0.812s, 128 output tokens, 157.60 tok/s
  Prompt 2: 0.813s, 128 output tokens, 157.52 tok/s
  Prompt 3: 0.812s, 128 output tokens, 157.72 tok/s
Run 3/3
  Prompt 1: 0.812s, 128 output tokens, 157.60 tok/s
  Prompt 2: 0.812s, 128 output tokens, 157.62 tok/s
  Prompt 3: 0.812s, 128 output tokens, 157.55 tok/s
{
  "avg_latency_sec": 0.816,
  "avg_tokens_per_sec": 156.9,
  "avg_completion_tokens": 128,
  "num_requests": 9
}
